In [37]:
# Plotting RNA-seq TPM across conditions with multiple replicates

Import packages, files

In [38]:
%pip install plotly
import pandas as pd
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

DEPRECATION: Loading egg at /Users/juliemcdonald/ENTER/lib/python3.12/site-packages/LoFreq_Star-2.1.5-py3.12.egg is deprecated. pip 24.3 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
Note: you may need to restart the kernel to use updated packages.


In [39]:
# --- Configuration ---
# List of (file_path, sample_name) tuples
files = [
    ("/Users/juliemcdonald/Documents/flamholz_lab/RNASeq_testing/palsson/palsson/lim_4h/final_gene_counts.csv", "L4_1"),
    ("/Users/juliemcdonald/Documents/flamholz_lab/RNASeq_testing/palsson/palsson/lim_4h_R2/final_gene_counts.csv", "L4_2"),
    ("/Users/juliemcdonald/Documents/flamholz_lab/RNASeq_testing/palsson/palsson/rep_4h/final_gene_counts.csv", "R4_1"),
    ("/Users/juliemcdonald/Documents/flamholz_lab/RNASeq_testing/palsson/palsson/rep_4h_R2/final_gene_counts.csv", "R4_2"),

]

# --- Load and merge ---
META_COLS = ["gene_id", "Chr", "Product", "Name"]

merged = None
for path, name in files:
    df = pd.read_csv(path)
    df = df.rename(columns={"TPM": f"{name}_TPM"})

    if merged is None:
        # First file: keep all metadata columns
        merged = df[META_COLS + [f"{name}_TPM"]]
    else:
        # Subsequent files: only bring in gene_id + TPM column
        merged = pd.merge(merged, df[["gene_id", f"{name}_TPM"]], on="gene_id", how="inner")

# --- Save ---
tpm_cols = [f"{name}_TPM" for _, name in files]
merged = merged[META_COLS + tpm_cols]

# --- Filter: keep only genes that reach cutoff TPM in at least one sample ---
cutoff = 5

high_expr_mask = (merged[tpm_cols] > cutoff).any(axis=1)
merged = merged[high_expr_mask]

merged.to_csv("/Users/juliemcdonald/Documents/flamholz_lab/RNASeq_testing/palsson/palsson/merged_gene_TPM_replicates.csv", index=False)
print(f"Done! {len(merged)} genes written to merged_gene_TPM.csv")

Done! 3759 genes written to merged_gene_TPM.csv


# Calculate data for plot
Average, standard deviation of replicates \\
Log-corrected standard deviation \\
RANSAC regression \\
RSD of linear fit

In [40]:
%pip install scikit-learn
from sklearn.linear_model import RANSACRegressor
from sklearn import linear_model
import plotly.express as px
import plotly.graph_objects as go

# Get all TPM columns
tpm_cols = [col for col in merged.columns if col.endswith('TPM')]

# Keep only genes with counts > 0 in all TPM columns
mask = (merged[tpm_cols] > 0).all(axis=1)
df = merged[mask].copy()

# Get all columns starting with L
l_cols = [col for col in df.columns if col.startswith('L')]

# Create average and standard deviation columns
df['L_avg'] = df[l_cols].mean(axis=1)
df['L_std'] = df[l_cols].std(axis=1)

# Get all columns starting with R
r_cols = [col for col in df.columns if col.startswith('R')]

# Create average and standard deviation columns
df['R_avg'] = df[r_cols].mean(axis=1)
df['R_std'] = df[r_cols].std(axis=1)

#Calculate relative error for symmetric error bars on log plot
#file:///Users/juliemcdonald/Downloads/EstimatingandPlottingLogarithmicErrorBars.pdf
df['R_rel_err'] = (df['R_std'] / df['R_avg']) / np.log(10)
df['L_rel_err'] = (df['L_std'] / df['L_avg']) / np.log(10)

#Choose what to plot
x_name = 'R_avg'
x_error = 'R_rel_err'
# x_error = 'R_std'

y_name = 'L_avg'
y_error = 'L_rel_err'
# y_error = 'L_std'

x = df[x_name]
y = df[y_name]
x_err = df[x_error]
y_err = df[y_error]

# Fit RANSAC regression on log10-transformed values
log_x = np.log10(x).values.reshape(-1, 1)
log_y = np.log10(y).values.reshape(-1, 1)

ransac = linear_model.RANSACRegressor(residual_threshold=1.0, random_state=100)
ransac.fit(log_x, log_y)

inlier_mask  = ransac.inlier_mask_
outlier_mask = np.logical_not(inlier_mask)

slope     = ransac.estimator_.coef_[0][0]
intercept = ransac.estimator_.intercept_[0]

# Generate fit line
x_fit = np.linspace(log_x.min(), log_x.max(), 200)
y_fit = slope * x_fit + intercept

#Calculate residuals & RSD threshold (on inliers only)
predicted        = slope * log_x.flatten() + intercept
residuals        = log_y.flatten() - predicted
rsd              = residuals[inlier_mask].std()
df["residual"]   = residuals

print(f"RANSAC slope={slope:.3f}, intercept={intercept:.3f}")
print(f"Inliers: {inlier_mask.sum()}  Outliers: {outlier_mask.sum()}")

outliers = df[outlier_mask].copy()
outliers['L/R'] = outliers['L_avg'] / outliers['R_avg']
outliers.to_csv('outliers.csv', index=False)
print(f"Done! {len(outliers)} genes written to outliers.csv")


DEPRECATION: Loading egg at /Users/juliemcdonald/ENTER/lib/python3.12/site-packages/LoFreq_Star-2.1.5-py3.12.egg is deprecated. pip 24.3 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
Note: you may need to restart the kernel to use updated packages.
RANSAC slope=0.820, intercept=0.121
Inliers: 3637  Outliers: 122
Done! 122 genes written to outliers.csv


# Plotting with plotly

In [41]:
# Convert fit line back to linear scale for plotly
line_X_linear = 10**x_fit
line_y_linear = 10**y_fit

inliers  = df[inlier_mask]
outliers = df[outlier_mask]

fig = go.Figure()

# Inliers
fig.add_trace(go.Scatter(
    x=inliers[x_name], y=inliers[y_name],
    error_x=dict(type='data', array=inliers[x_error].values, visible=True),
    error_y=dict(type='data', array=inliers[y_error].values, visible=True),
    mode='markers',
    marker=dict(size=6, color='silver', opacity=0.6,
                line=dict(width=2, color='DarkSlateGrey')),
    name='Inliers',
    customdata=inliers[['Name', 'gene_id', 'Product', 'Chr', x_name, y_name]].values,
    hovertemplate=(
        "<b>%{customdata[0]}</b><br>"
        "gene_id: %{customdata[1]}<br>"
        "Product: %{customdata[2]}<br>"
        "Chr: %{customdata[3]}<br>"
        f"{x_name}: " + "%{customdata[4]:.2f}<br>"
        f"{y_name}: " + "%{customdata[5]:.2f}<extra></extra>"
    )
))

# Outliers
fig.add_trace(go.Scatter(
    x=outliers[x_name], y=outliers[y_name],
    error_x=dict(type='data', array=outliers[x_error].values, visible=True),
    error_y=dict(type='data', array=outliers[y_error].values, visible=True),
    mode='markers',
    marker=dict(size=6, color='darkorange', opacity=0.6,
                line=dict(width=2, color='DarkSlateGrey')),
    name='Outliers',
    customdata=outliers[['Name', 'gene_id', 'Product', 'Chr', x_name, y_name]].values,
    hovertemplate=(
        "<b>%{customdata[0]}</b><br>"
        "gene_id: %{customdata[1]}<br>"
        "Product: %{customdata[2]}<br>"
        "Chr: %{customdata[3]}<br>"
        f"{x_name}: " + "%{customdata[4]:.2f}<br>"
        f"{y_name}: " + "%{customdata[5]:.2f}<extra></extra>"
    )
))

# RANSAC fit line
fig.add_trace(go.Scatter(
    x=line_X_linear, y=line_y_linear,
    mode='lines',
    line=dict(color='orange', width=2),
    name=f'RANSAC fit (slope={slope:.2f}, intercept={intercept:.2f})'
))

fig.update_layout(
    width=750, height=620,
    plot_bgcolor='white',
    title=f"R vs L gene TPM — {len(df):,} genes. Cutoff = {cutoff} TPM",
    xaxis=dict(type='log', showgrid=True, gridcolor='#eeeeee', title=x_name),
    yaxis=dict(type='log', showgrid=True, gridcolor='#eeeeee', title=y_name),
)

fig.show()
fig.write_html("scatter_wt_vs_lim.html")
print("Saved: scatter_wt_vs_lim.html")


Saved: scatter_wt_vs_lim.html


# Optional: add lines to show residual error in fit

In [42]:
# Optional: add ±2 RSD envelope lines around RANSAC fit

for label, offset, dash in [
    ("+2 RSD", 2*rsd, "dot"),
    ("-2 RSD", -2*rsd, "dot"),
]:
    fig.add_scatter(
        x=10**x_fit,
        y=10**(y_fit + offset),
        mode="lines",
        line=dict(color="orange", width=1, dash=dash),
        name=label,
    )

fig.show()
fig.write_html("scatter_wt_vs_lim.html")
print("Saved: scatter_wt_vs_lim.html")


Saved: scatter_wt_vs_lim.html
